In [ ]:
from dataclasses import dataclass

from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False  
GRPC_HOST = "localhost:50052"

### Message type

In [ ]:
@dataclass
class Message:
    content: str

### gRPC host

In [ ]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address=GRPC_HOST)
host.start()

### Routed agents (style, security, lead)

In [ ]:
class StyleReviewer(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[],
            system_message=(
                "You review code for readability, naming, structure, and maintainability. "
                "Short bullets; mark info / warn / important. No security topics."
            ),
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class SecurityReviewer(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[],
            system_message=(
                "You review code for security: injection, secrets, unsafe IO, weak crypto, "
                "command execution, auth issues, etc. Short bullets; severity low/medium/high. No style topics."
            ),
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class LeadReviewer(RoutedAgent):
    def __init__(self, name: str, language: str = "python") -> None:
        super().__init__(name)
        self._language = language
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[],
            system_message=(
                "You synthesize specialist reviews only—do not invent new issues. "
                "Output: (1) executive summary, (2) merged findings by theme, "
                "(3) one recommendation: approve with nits / request changes / block."
            ),
        )

    def _wrap_code(self, code: str) -> str:
        return f"```{self._language}\n{code.strip()}\n```"

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        code = message.content.strip()
        block = self._wrap_code(code)
        style_prompt = "Review for STYLE and maintainability only.\n\n" + block
        security_prompt = "Review for SECURITY only.\n\n" + block
        r_style = await self.send_message(Message(content=style_prompt), AgentId("style_reviewer", "default"))
        r_sec = await self.send_message(Message(content=security_prompt), AgentId("security_reviewer", "default"))
        bundle = f"## Style review\n{r_style.content}\n\n## Security review\n{r_sec.content}\n"
        merge_prompt = "Specialist reviews follow. Synthesize per your system message.\n\n" + bundle
        tm = TextMessage(content=merge_prompt, source="user")
        synth = await self._delegate.on_messages([tm], ctx.cancellation_token)
        return Message(content=bundle + "\n## Lead summary\n\n" + synth.chat_message.content)

### Register workers

In [ ]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:
    worker = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker.start()
    await StyleReviewer.register(worker, "style_reviewer", lambda: StyleReviewer("style_reviewer"))
    await SecurityReviewer.register(worker, "security_reviewer", lambda: SecurityReviewer("security_reviewer"))
    await LeadReviewer.register(worker, "lead_reviewer", lambda: LeadReviewer("lead_reviewer"))
    agent_id = AgentId("lead_reviewer", "default")
else:
    w_style = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await w_style.start()
    await StyleReviewer.register(w_style, "style_reviewer", lambda: StyleReviewer("style_reviewer"))

    w_sec = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await w_sec.start()
    await SecurityReviewer.register(w_sec, "security_reviewer", lambda: SecurityReviewer("security_reviewer"))

    worker = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker.start()
    await LeadReviewer.register(worker, "lead_reviewer", lambda: LeadReviewer("lead_reviewer"))
    agent_id = AgentId("lead_reviewer", "default")

### Run review

In [ ]:
from pathlib import Path

CODE_PATH = Path.cwd() / "sample_review_target.py"

if not CODE_PATH.is_file():
    raise FileNotFoundError(
        f"Expected {CODE_PATH.resolve()} to be present in the current working directory"
    )

SAMPLE_CODE = CODE_PATH.read_text(encoding="utf-8")
print(f"Reviewing: {CODE_PATH.resolve()} ({len(SAMPLE_CODE)} chars)")

response = await worker.send_message(Message(content=SAMPLE_CODE), agent_id)
display(Markdown(response.content))

### Shutdown


In [ ]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await w_style.stop()
    await w_sec.stop()

await host.stop()